In [2]:
from bs4 import BeautifulSoup, Tag
from datetime import datetime
from tqdm import tqdm
import pandas as pd
import requests
from utils.helper_functions import get_path


In [ ]:
def get_screener_url(company_or_ticker: str) -> str:
    # Using screener api to get the exact url to hit for the required company
    results = requests.get(
        f"https://www.screener.in/api/company/search/?q={company_or_ticker}"
    )
    valid_results = [
        res for res in results.json()
        if not any(term in res["name"].lower() for term in ["partly", "paid", "rights", "warrant", "re-"])
    ]
    
    if not valid_results:
        print(f"No valid company found for ticker: {company_or_ticker}")
        return ''

    primary_company = valid_results[0]
    primary_company_path = primary_company['url']
    complete_screener_url = f"https://www.screener.in{primary_company_path}"
    
    return complete_screener_url

def latest_qtr_result(screener_url: str) -> pd.DataFrame:
    response = requests.get(screener_url)

    soup = BeautifulSoup(response.text)

    qty_res_table_soup = soup.find_all('table', class_="data-table responsive-text-nowrap")

    dfs = []  # list to store all dataframes

    for table in qty_res_table_soup:
        # Extract headers
        headers = [th.get_text(strip=True) for th in table.find_all("th")]
        
        # Extract all rows
        rows = []
        for tr in table.find_all("tr"):
            cells = tr.find_all(["td"])
            if not cells:
                continue
            row = [td.get_text(strip=True) for td in cells]
            rows.append(row)
        
        # Create DataFrame
        df = pd.DataFrame(rows, columns=headers)
        print(df.shape)
        dfs.append(df)

    # --- optional: unpack them for clarity ---
    quarterly_df = dfs[0]
    annual_df = dfs[1]

    return quarterly_df

In [7]:
out = get_screener_url('pvv')
out

'https://www.screener.in/company/536659/'

In [8]:
output = latest_qtr_result(out)
output

(12, 14)
(12, 14)
(10, 13)
(6, 13)
(6, 13)


ValueError: 12 columns passed, passed data had 46 columns